# Phase 4 — The Modeling Table: Six Features, Measured One Quarter Earlier

**Purpose.** Build `quarterly_modeling_table`: one row per county and label quarter, holding the decline label and six features. `sql/005_phase4_features.sql` builds it on the server, and `sql/005b_cv_fold_rows.sql` then rebuilds the fold view that depends on it. This notebook runs both, checks the result, and **recomputes every feature independently in pandas** for a random sample of rows.

**The six features**

| Feature | Definition |
|---|---|
| `momentum_1m`, `momentum_3m`, `momentum_6m` | ZHVI change over 1, 3 and 6 months, at the quarter-end month of the feature quarter |
| `rate_level` | The 30-year mortgage rate at that quarter end |
| `rate_roc_12m` | The 12-month change of that rate, in percentage points |
| `rate_momentum_interaction` | `rate_roc_12m` x `momentum_3m`. Three months is one quarter, the horizon being predicted |

**Rules**

- **No look-ahead.** Every feature attached to a label is measured at the end of the quarter **before** the one being predicted. Rows are dated by their label quarter, and `feature_quarter_end` records the date the features were measured.
- **Both quarter-end months must have a value**, as for the label in notebook 03.
- **Any missing feature drops the row**, so no model input is ever empty.
- **All splits are built.** Unlike notebook 03, this runs on Train, Embargo and Test: Test rows need real feature values too. What must never happen is a Test row informing a modeling *decision*.

## 1. Setup and connection

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_sql")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


def run_sql_file(name):
    """Run a whole SQL file through the raw database cursor, with no parameters."""
    path = Path("../sql") / name
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute(path.read_text(encoding="utf-8"))
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    print(f"ran {path}")


os.makedirs("../outputs", exist_ok=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())
    for name in ("monthly_split", "cv_folds", "mortgage_monthly"):
        assert conn.execute(text("SELECT to_regclass(:n)"), {"n": name}).scalar(), (
            f"STOP: {name} not found. Run notebooks 01 and 02 first.")
print("monthly_split, cv_folds and mortgage_monthly found.")

capstone_sql_variant
monthly_split, cv_folds and mortgage_monthly found.


## 2. Build the modeling table and the fold view

`005` drops and rebuilds its tables, and dropping `quarterly_modeling_table` also drops the view `cv_fold_rows`, so `005b` has to follow right after it. Both are safe to re-run.

In [2]:
run_sql_file("005_phase4_features.sql")
run_sql_file("005b_cv_fold_rows.sql")

split_summary = pd.read_sql(
    """
    SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT county_fips) AS n_counties,
           ROUND(100.0 * AVG(decline_label), 2) AS positive_rate_pct
    FROM quarterly_modeling_table GROUP BY split ORDER BY split
    """, engine)
print(f"Total: {int(split_summary['n_rows'].sum()):,} rows")
split_summary

ran ..\sql\005_phase4_features.sql
ran ..\sql\005b_cv_fold_rows.sql
Total: 222,530 rows


,split,n_rows,n_counties,positive_rate_pct
0,embargo,3020,3020,0.96
1,test,51980,3071,8.11
2,train,164460,3009,4.85
3,unassigned,3070,3070,4.27


**Expected with the Zillow file of October 2026:**

| Split | Rows | Counties | Decline rate |
|---|---|---|---|
| embargo | 3,020 | 3,020 | 0.96% |
| test | 51,980 | 3,071 | 8.11% |
| train | 164,460 | 3,009 | 4.85% |
| unassigned | 3,070 | 3,070 | 4.27% |

The modeling table has fewer Train rows than the label table of notebook 03 (164,460 against 173,277), because it also drops each county's earliest quarters, where the 6-month momentum or the 12-month rate history does not exist yet. Unassigned is one quarter, 2026-Q2.

## 3. Sanity checks on the table

In [3]:
FEATURES = ["momentum_1m", "momentum_3m", "momentum_6m", "rate_level", "rate_roc_12m", "rate_momentum_interaction"]
mt = pd.read_sql("SELECT * FROM quarterly_modeling_table", engine)
mt["quarter"] = pd.to_datetime(mt["quarter"])
mt["feature_quarter_end"] = pd.to_datetime(mt["feature_quarter_end"])
mt["label_quarter_end"] = pd.to_datetime(mt["label_quarter_end"])

n_missing = int(mt[FEATURES].isna().sum().sum())
n_dup = int(mt.duplicated(["county_fips", "quarter"]).sum())
n_inter = int(((mt["rate_momentum_interaction"] - mt["rate_roc_12m"] * mt["momentum_3m"]).abs() > 1e-9).sum())
n_month = int((~mt["feature_quarter_end"].dt.month.isin([3, 6, 9, 12])).sum())
n_misaligned = int((mt["feature_quarter_end"] != mt["quarter"] - pd.Timedelta(days=1)).sum())

print(f"Shape: {mt.shape}")
print(f"Missing feature values: {n_missing} | duplicate county-quarters: {n_dup} | "
      f"wrong interaction: {n_inter} | feature date not a quarter end: {n_month} | misaligned: {n_misaligned}")
assert n_missing == 0 and n_dup == 0 and n_inter == 0 and n_month == 0 and n_misaligned == 0, "A sanity check failed."
print("No missing features, one row per county and quarter, and the interaction is the plain product.")

Shape: (222530, 12)
Missing feature values: 0 | duplicate county-quarters: 0 | wrong interaction: 0 | feature date not a quarter end: 0 | misaligned: 0
No missing features, one row per county and quarter, and the interaction is the plain product.


## 4. Feature summary on Train

Scale and range of each feature on the Train rows. The summary is saved for the modeling phase.

In [4]:
train = mt[mt["split"] == "train"]
summary = train[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]]
summary.to_csv("../outputs/phase4_feature_summary.csv")
summary.round(4)

,mean,std,min,50%,max
momentum_1m,0.0034,0.0071,-0.0810,0.0031,0.0730
momentum_3m,0.0100,0.0197,-0.2018,0.0096,0.2284
momentum_6m,0.0197,0.0360,-0.3058,0.0191,0.3738
rate_level,4.4938,1.0825,2.6840,4.1960,7.1600
rate_roc_12m,-0.2171,0.5813,-1.4200,-0.2480,1.1125
rate_momentum_interaction,-0.0019,0.0139,-0.2200,-0.0004,0.1776


## 5. Independent check: recompute every feature in pandas

The SQL builds the features with window functions. Here a random sample of 2,000 rows is recomputed **from the raw monthly tables in pandas**, and compared with the table:

- the three momentum values, from the ZHVI at the feature quarter end and 1, 3 and 6 months earlier;
- the rate level and the 12-month rate change, from the monthly rate;
- the label, from the ZHVI at the label quarter end against the feature quarter end.

Any difference stops the notebook. This is a check of the SQL by a second implementation, not a repeat of it.

In [5]:
rng = np.random.default_rng(42)
sample = mt.sample(n=2000, random_state=42).reset_index(drop=True)

z = pd.read_sql("SELECT county_fips, month, zhvi FROM monthly_split", engine)
z["month"] = pd.to_datetime(z["month"])
zhvi = z.set_index(["county_fips", "month"])["zhvi"]
mortgage = pd.read_sql("SELECT month, mortgage_rate FROM mortgage_monthly", engine)
mortgage["month"] = pd.to_datetime(mortgage["month"])
rate = mortgage.set_index("month")["mortgage_rate"]


def month_end(dates, shift_months=0):
    return (pd.to_datetime(dates).dt.to_period("M") - shift_months).dt.to_timestamp(how="end").dt.normalize()


def zhvi_at(df, date_col, shift_months=0):
    keys = pd.MultiIndex.from_arrays([df["county_fips"], month_end(df[date_col], shift_months)])
    return zhvi.reindex(keys).to_numpy()


z0 = zhvi_at(sample, "feature_quarter_end")
recomputed = pd.DataFrame({
    "momentum_1m": z0 / zhvi_at(sample, "feature_quarter_end", 1) - 1,
    "momentum_3m": z0 / zhvi_at(sample, "feature_quarter_end", 3) - 1,
    "momentum_6m": z0 / zhvi_at(sample, "feature_quarter_end", 6) - 1,
    "rate_level": rate.reindex(month_end(sample["feature_quarter_end"])).to_numpy(),
})
recomputed["rate_roc_12m"] = recomputed["rate_level"] - rate.reindex(month_end(sample["feature_quarter_end"], 12)).to_numpy()
recomputed["rate_momentum_interaction"] = recomputed["rate_roc_12m"] * recomputed["momentum_3m"]
recomputed["decline_label"] = (zhvi_at(sample, "label_quarter_end") / z0 - 1 < -0.02).astype(int)

diffs = {f: float((recomputed[f] - sample[f]).abs().max()) for f in FEATURES}
label_mismatch = int((recomputed["decline_label"] != sample["decline_label"]).sum())
print("Largest absolute difference per feature, SQL against pandas:")
for f, d in diffs.items():
    print(f"  {f:28s} {d:.2e}")
print(f"Label mismatches: {label_mismatch} of {len(sample):,}")
assert max(diffs.values()) < 1e-9, "A feature differs from the independent recomputation."
assert label_mismatch == 0, "A label differs from the independent recomputation."
print("All six features and the label match the independent recomputation.")

Largest absolute difference per feature, SQL against pandas:
  momentum_1m                  0.00e+00
  momentum_3m                  0.00e+00
  momentum_6m                  0.00e+00
  rate_level                   0.00e+00
  rate_roc_12m                 0.00e+00
  rate_momentum_interaction    0.00e+00
Label mismatches: 0 of 2,000
All six features and the label match the independent recomputation.


## 6. The validation folds on the new table

The fold view `cv_fold_rows` was just rebuilt on this table. Every validation fold needs at least 20 declines before its score is trusted, and no Embargo or Test row may enter a fold.

In [6]:
fold_summary = pd.read_sql(
    """
    SELECT fold_id, fold_role, COUNT(*) AS n_rows, SUM(decline_label) AS n_positives,
           ROUND(100.0 * AVG(decline_label), 2) AS positive_rate_pct
    FROM cv_fold_rows GROUP BY fold_id, fold_role ORDER BY fold_id, fold_role
    """, engine)
leak = pd.read_sql("SELECT COUNT(*) FROM cv_fold_rows WHERE split <> 'train'", engine).iloc[0, 0]
val = fold_summary[fold_summary["fold_role"] == "val"]
lowest = val.loc[val["n_positives"].idxmin()]
print(f"{len(val)} validation folds | lowest: fold {int(lowest['fold_id'])} with {int(lowest['n_positives'])} declines | "
      f"fold rows outside Train: {int(leak)}")
print(f"Pooled validation: {int(val['n_rows'].sum()):,} rows, {int(val['n_positives'].sum()):,} declines, "
      f"{100 * val['n_positives'].sum() / val['n_rows'].sum():.2f}%")
assert val["n_positives"].min() >= 20, "A validation fold has fewer than 20 declines."
assert leak == 0, "A fold row comes from Embargo or Test."
fold_summary

11 validation folds | lowest: fold 10 with 71 declines | fold rows outside Train: 0
Pooled validation: 112,734 rows, 4,252 declines, 3.77%


,fold_id,fold_role,n_rows,n_positives,positive_rate_pct
0,0,train,42722,3526,8.25
1,0,val,7892,882,11.18
2,1,train,50614,4408,8.71
3,1,val,8232,1121,13.62
4,2,train,58846,5529,9.40
5,2,val,8683,288,3.32
6,3,train,67529,5817,8.61
7,3,val,9667,301,3.11
8,4,train,77196,6118,7.93
9,4,val,9874,325,3.29


**Expected with the Zillow file of October 2026:** 11 validation folds, the lowest being fold 10 (validation year 2020) with 71 declines. Pooled across the folds, validation holds 112,734 rows and 4,252 declines, which is 3.77%. The early folds have high decline rates (11% in 2010 and nearly 14% in 2011) and the later ones low rates (1.6% in 2015 and 0.6% in 2020), so one pooled score would hide that. Results will be reported per fold.

## Status

Phase 4 is complete when the split counts match the expected figures, the sanity checks and the independent recomputation pass, and the fold checks pass. `quarterly_modeling_table` and the view `cv_fold_rows` are the direct input of the modeling phase: no further joining or reshaping is needed.